# Agentic Fraud Investigation: From Alert to Grounded Recommendation

## 1. Why ML Scoring Is Not Enough

The model answers **“How risky is this transaction?”** The investigation workflow answers **“Why does this alert deserve investigation?”** A human analyst answers **“What action should we actually take?”**

This notebook demonstrates tool selection, evidence retrieval, state, policy grounding, and an explicit workflow without requiring an LLM or paid API.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.fraud_tools import FraudTools
from src.investigation_agent import InvestigationAgent

tools = FraudTools(PROJECT_ROOT)
agent = InvestigationAgent(PROJECT_ROOT)

## 2. Fraud Investigation Architecture

`Alert → transaction lookup → customer history → suspicious signals → policy retrieval → evidence summary → CLOSE / REVIEW / ESCALATE → human decision`

The workflow is deliberately fixed and auditable. An LLM could replace only the final synthesis layer in production.

## 3. Available Agent Tools

1. `get_transaction(transaction_id)` — transaction, model score, and model-visible features.
2. `get_customer_history(customer_id, cutoff)` — history strictly before the alert.
3. `search_fraud_policy(query, top_k=3)` — TF-IDF + cosine similarity over policy sections.
4. `get_risk_score(transaction_id)` — probability, threshold, and display risk level.

## 4. Policy Retrieval

The small synthetic policy is split on Markdown headings. TF-IDF is transparent and reproducible; a production system could replace it with evaluated hybrid/vector retrieval.

In [2]:
display(pd.DataFrame(tools.search_fraud_policy(
    "international high amount new geography rapid transactions new device", top_k=3
))[["section", "similarity", "citation"]])

,section,similarity,citation
0,Account Takeover Indicators,0.413,[Policy: Account Takeover Indicators]
1,New Device,0.204,[Policy: New Device]
2,International Activity,0.176,[Policy: International Activity]


## 5. Investigation Example

Choose a high-scoring test alert with at least three observable signals. The true label is intentionally excluded from the investigation call.

In [3]:
scores = pd.read_csv(PROJECT_ROOT / "data" / "sample" / "scored_test_transactions.csv")
candidates = scores.loc[(scores.risk_level == "HIGH") & (scores.signal_count >= 3)]
if candidates.empty:
    candidates = scores.loc[scores.risk_level == "HIGH"]
demo_transaction_id = candidates.sort_values("fraud_probability", ascending=False).iloc[0].transaction_id
transaction = tools.get_transaction(demo_transaction_id)
display(pd.Series(transaction, name="alert").to_frame())

,alert
transaction_id,TX_074381
customer_id,CUST_07431
transaction_timestamp,2025-12-12T21:48:17
amount,478.93
merchant_category,fuel
merchant_country,MA
transaction_channel,mobile
device_id,DEV_X103771
is_card_present,0
is_international,1


## 6. Evidence Collected

In [4]:
history = tools.get_customer_history(transaction["customer_id"], transaction["transaction_timestamp"])
display(pd.Series({k: v for k, v in history.items() if k != "recent_transactions"}, name="history").to_frame())
display(pd.DataFrame(history["recent_transactions"]))

,history
customer_id,CUST_07431
history_cutoff,2025-12-12T21:48:17
transaction_count,16
historical_average_amount,139.42
historical_max_amount,760.98
historical_international_rate,0.125
known_countries,"[CA, GB]"
known_devices,3
merchant_categories,"[cash_withdrawal, dining, electronics, fuel, g..."
transactions_previous_1h,0


,transaction_id,transaction_timestamp,amount,merchant_country,merchant_category,transaction_channel,device_id
0,TX_074373,2025-08-10T14:04:33,36.94,GB,fuel,pos,DEV_07431_A
1,TX_074374,2025-09-06T04:17:00,75.61,CA,grocery,online,DEV_07431_A
2,TX_074375,2025-09-07T16:56:07,66.59,GB,retail,atm,DEV_07431_A
3,TX_074376,2025-10-01T11:07:20,22.17,GB,fuel,mobile,DEV_X083733
4,TX_074377,2025-10-01T11:23:03,44.25,GB,fuel,pos,DEV_07431_A
5,TX_074378,2025-10-26T04:45:36,35.10,GB,grocery,pos,DEV_07431_A
6,TX_074379,2025-11-07T13:16:51,22.22,GB,fuel,pos,DEV_07431_A
7,TX_074380,2025-12-05T10:56:32,168.32,GB,electronics,pos,DEV_07431_A


## 7–8. Policy Citations & Agent Recommendation

In [5]:
result = agent.investigate(demo_transaction_id)
print(json.dumps(result, indent=2))
assert result["tool_trace"] == ["get_transaction", "get_customer_history", "search_fraud_policy", "get_risk_score"]
assert all(item["citation"].startswith("[Policy: ") for item in result["retrieved_policies"])
assert "definitely fraudulent" not in json.dumps(result).lower()

{
  "transaction_id": "TX_074381",
  "fraud_probability": 0.9005,
  "selected_threshold": 0.5,
  "risk_level": "HIGH",
  "signals": [
    "Amount $478.93 is 3.4x the customer's prior average.",
    "International activity is unusual relative to the 12.5% prior international rate.",
    "Merchant country MA is not present in prior customer history.",
    "The device was not seen in the customer's prior transactions."
  ],
  "retrieved_policies": [
    {
      "section": "Account Takeover Indicators",
      "citation": "[Policy: Account Takeover Indicators]",
      "similarity": 0.408
    },
    {
      "section": "Geographic Anomaly",
      "citation": "[Policy: Geographic Anomaly]",
      "similarity": 0.342
    },
    {
      "section": "New Device",
      "citation": "[Policy: New Device]",
      "similarity": 0.251
    }
  ],
  "recommendation": "ESCALATE",
  "reason": "The score exceeds the review threshold and multiple independent signals justify prompt analyst review.",
  "requir

## 9. Human-in-the-Loop

`CLOSE`, `REVIEW`, and `ESCALATE` are recommendations. Only authorized humans may block an account, reverse a payment, freeze funds, contact a customer, or make a regulatory escalation.

## 10. Failure Modes

- **False positives:** legitimate travel, device changes, or large purchases may look unusual.
- **False negatives:** fraud may resemble normal behavior.
- **Retrieval failure:** lexical matching may miss the best policy section.
- **Unsupported claims:** synthesis might say more than the tools returned.
- **Stale policy / model drift:** policy and fraud behavior change.

Mitigations include grounded summaries, citations, human approval, retrieval evaluation, versioned policy, audit logs, and model monitoring.

## 11. Monitoring

- **Model:** PR-AUC, precision, recall, false-positive/negative rates, calibration, score distribution.
- **Operations:** alerts per 1,000, analyst workload, latency, escalation percentage.
- **Agent/RAG:** tool success, retrieval relevance, citation correctness, unsupported-claim rate.
- **Business:** captured fraud, losses, customer friction, false declines, resolution time.

## 12. Key Takeaways

The agent uses tools instead of receiving all context upfront, cites retrieved evidence, and produces a reproducible recommendation. It does not claim certainty and cannot execute high-impact actions.